In [1]:
import sys, os
sys.path.insert(0, os.path.join('..'))
import numpy as np
import pandas as pd

Benchmark data set

## Efficient Frontier Generation
The following cells execute the C++ PBIL engine across a range of target returns ($\mu$) to generate the Efficient Frontier, then parse the output to extract normalized CVaR and portfolio weights.

In [4]:
import os
import subprocess
import re
import matplotlib.pyplot as plt

def run_pbil(exe_path, data_path, total_assets, cardinality, pop_size, mu, initial_prices_str):
    command = [
        exe_path,
        data_path,
        str(total_assets),
        str(cardinality),
        str(pop_size),
        str(mu),
        initial_prices_str,
    ]
    try:
        result = subprocess.run(command, check=True, stdout=subprocess.PIPE, text=True)
        match = re.search(r"FINAL_CVAR:([0-9.eE+-]+)", result.stdout)
        k_match = re.search(r"PORTFOLIO_K:([0-9,]+)", result.stdout)
        w_match = re.search(r"PORTFOLIO_W:([0-9.eE+-,]+)", result.stdout)

        if match and k_match and w_match:
            cvar = float(match.group(1)) / 100000.0
            k_indices = [int(x) for x in k_match.group(1).split(',')]
            weights = [float(x) for x in w_match.group(1).split(',')]
            return cvar, k_indices, weights
        return None, None, None
    except subprocess.CalledProcessError as e:
        print("C++ Engine Error:", e.returncode)
        return None, None, None

In [5]:
tickers = [
    "AAPL", "ABBV", "ABT", "ACN", "ADBE", "AMAT", "AMD", "AMGN", "AMT", "AMZN",
    "AVGO", "AXP", "BA", "BAC", "BK", "BKNG", "BLK", "BMY", "BRK-B", "C",
    "CAT", "CL", "CMCSA", "COF", "COP", "COST", "CRM", "CSCO", "CVS", "CVX",
    "DE", "DHR", "DIS", "DUK", "EMR", "FDX", "GD", "GE", "GILD",
    "GM", "GOOG", "GOOGL", "GS", "HD", "HON", "IBM", "INTC", "INTU", "ISRG",
    "JNJ", "JPM", "KO", "LIN", "LLY", "LMT", "LOW", "LRCX", "MA", "MCD",
    "MDLZ", "MDT", "META", "MMM", "MO", "MRK", "MS", "MSFT", "MU", "NEE",
    "NFLX", "NKE", "NOW", "NVDA", "ORCL", "PEP", "PFE", "PG", "PM",
    "QCOM", "RTX", "SBUX", "SCHW", "SO", "SPG", "T", "TMO", "TMUS", "TSLA",
    "TXN", "UNH", "UNP", "UPS", "USB", "V", "VZ", "WFC", "WMT", "XOM"
]

prices = pd.DataFrame({
    t: np.load(f"../data/raw/{t}.npy") for t in tickers
})
initial_prices = prices.iloc[0].to_numpy()
initial_prices_str = ",".join(map(str, initial_prices))

csv_file = "../data/scenarios/tech_scenarios.csv"
exe_file = "../build/Release/optimizer_app.exe"

Q = len(tickers)
K = 10
POPULATION = 200

mu_values = np.linspace(0.001, 0.015, 15)
frontier = []

print("=== Generating Efficient Frontier ===")
for mu in mu_values:
    print(f"Running optimization for target return: {mu:.4f}...")
    cvar_norm, best_k, best_w = run_pbil(exe_file, csv_file, Q, K, POPULATION, mu, initial_prices_str)
    if cvar_norm is not None:
        asset_weights = []
        for idx, shares in zip(best_k, best_w):
            ticker = tickers[idx]
            price = initial_prices[idx]
            weight = (shares * price) / 100000.0
            asset_weights.append((ticker, weight))
        
        frontier.append((mu, cvar_norm, asset_weights))
        print(f"  -> Normalized CVaR = {cvar_norm:.6f}")

=== Generating Efficient Frontier ===
Running optimization for target return: 0.0010...
C++ Engine Error: 3221226505
Running optimization for target return: 0.0020...
C++ Engine Error: 3221226505
Running optimization for target return: 0.0030...
C++ Engine Error: 3221226505
Running optimization for target return: 0.0040...
C++ Engine Error: 3221226505
Running optimization for target return: 0.0050...
C++ Engine Error: 3221226505
Running optimization for target return: 0.0060...
C++ Engine Error: 3221226505
Running optimization for target return: 0.0070...
C++ Engine Error: 3221226505
Running optimization for target return: 0.0080...
C++ Engine Error: 3221226505
Running optimization for target return: 0.0090...
C++ Engine Error: 3221226505
Running optimization for target return: 0.0100...
C++ Engine Error: 3221226505
Running optimization for target return: 0.0110...
C++ Engine Error: 3221226505
Running optimization for target return: 0.0120...
C++ Engine Error: 3221226505
Running optimi

In [4]:
if frontier:
    mu_list = [f[0] for f in frontier]
    cvar_list = [f[1] for f in frontier]

    plt.figure(figsize=(10, 6))
    plt.plot(cvar_list, mu_list, marker='o', linestyle='-', color='b', linewidth=2, markersize=8)
    plt.title('Efficient Frontier (PBIL Optimizer)', fontsize=14)
    plt.xlabel('Normalized CVaR (Risk)', fontsize=12)
    plt.ylabel('Target Return ($\mu$)', fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.7)

    for mu, cvar, _ in frontier:
        plt.annotate(f"{mu:.3f}", (cvar, mu), textcoords="offset points", xytext=(10,-5), ha='left')

    plt.tight_layout()
    plt.show()

    print("\n=== Portfolio Allocations along the Frontier ===")
    for mu, cvar, asset_weights in frontier:
        print(f"Target Return: {mu:.4f} | Normalized CVaR: {cvar:.6f}")
        for t, w in asset_weights:
            if w > 1e-4:
                print(f"  {t}: {w*100:.2f}%")
        print("-" * 40)
else:
    print("No points successfully generated to plot.")

No points successfully generated to plot.


<>:9: SyntaxWarning: invalid escape sequence '\m'
<>:9: SyntaxWarning: invalid escape sequence '\m'
C:\Users\willi\AppData\Local\Temp\ipykernel_8564\2879402988.py:9: SyntaxWarning: invalid escape sequence '\m'
  plt.ylabel('Target Return ($\mu$)', fontsize=12)
